# Natural Language Understanding - Homework 1
# Text Classification: Naive Bayes and Logistic Regression

**Student Name:** [Your Name Here]

**Date:** March 2, 2026

## Collaborators and Resources

**Collaborators:** None [or list names]

**Resources Used:**
- Scikit-learn documentation: https://scikit-learn.org/
- NumPy documentation: https://numpy.org/
- Pandas documentation: https://pandas.pydata.org/
- SLP textbook Chapter 4

---
# Part 0: Paper Exercise

## Exercise 4.2 from SLP (Hand Calculation)

**[COMPLETE THIS BY HAND AND INCLUDE YOUR WORK HERE]**

You can either:
1. Write your solution in markdown/LaTeX here
2. Scan/photograph your handwritten work and insert the image
3. Include it as a separate PDF

Make sure to show:
- Prior probabilities P(c) for each class
- Likelihood probabilities P(word|c) with add-1 smoothing
- Posterior calculations for the test document
- Final classification decision

---
# Part 1: Preparing Data

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
import re
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully!")

## Step 1: Load the Data

In [ ]:
# Load the dataset
df = pd.read_csv('Tweets_5K.csv')
print("Dataset loaded successfully!")
print(f"\nDataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
df.info()

In [ ]:
# Create list of raw tweets and labels
raw_tweets = df['text'].tolist()

# Map sentiments to integers
sentiment_mapping = {
    'negative': -1,
    'neutral': 0,
    'positive': 1
}

labels = df['sentiment'].map(sentiment_mapping).tolist()

print(f"Number of tweets: {len(raw_tweets)}")
print(f"Number of labels: {len(labels)}")
print(f"\nLabel distribution:")
print(pd.Series(labels).value_counts().sort_index())
print(f"\nExample tweet: {raw_tweets[0]}")
print(f"Example label: {labels[0]}")

## Step 2: Basic Preprocessing (Tokenization)

We'll create a simple tokenizer that splits on whitespace only.

In [ ]:
# Basic preprocessing: split on whitespace only
basic_preproc_tweets = [tweet.split() for tweet in raw_tweets]

print(f"Number of preprocessed tweets: {len(basic_preproc_tweets)}")
print(f"\nOriginal tweet 0: {raw_tweets[0]}")
print(f"Preprocessed tweet 0: {basic_preproc_tweets[0]}")
print(f"\nOriginal tweet 1: {raw_tweets[1]}")
print(f"Preprocessed tweet 1: {basic_preproc_tweets[1]}")

## Step 3: Featurize (Manual Bag of Words)

**IMPORTANT:** We are NOT using CountVectorizer as per assignment instructions.
Instead, we'll manually create the bag-of-words matrix using DictVectorizer.

In [ ]:
# Create word count dictionaries for each tweet
def create_word_count_dict(tweet_tokens):
    """
    Create a dictionary of word counts from a list of tokens.
    
    Args:
        tweet_tokens: List of word tokens
    
    Returns:
        Dictionary mapping word -> count
    """
    word_count = {}
    for word in tweet_tokens:
        word_count[word] = word_count.get(word, 0) + 1
    return word_count

# Create list of dictionaries
tweet_dicts = [create_word_count_dict(tweet) for tweet in basic_preproc_tweets]

print(f"Created {len(tweet_dicts)} word count dictionaries")
print(f"\nExample dictionary for tweet 0:")
print(tweet_dicts[0])

In [ ]:
# Use DictVectorizer to create bag-of-words matrix
vectorizer = DictVectorizer(sparse=False)
basic_preproc_bow = vectorizer.fit_transform(tweet_dicts)

# Get feature names
feature_names = vectorizer.get_feature_names_out()

print(f"Bag-of-words matrix shape: {basic_preproc_bow.shape}")
print(f"Number of documents (tweets): {basic_preproc_bow.shape[0]}")
print(f"Number of features (unique words): {basic_preproc_bow.shape[1]}")
print(f"\nFirst 10 features: {feature_names[:10]}")

## Step 4: Create Training and Test Sets

We'll use an 80/20 split. Since the data is pre-shuffled, we'll take the first 80% as training and last 20% as test.

In [ ]:
# Calculate split index
split_index = int(0.8 * len(basic_preproc_bow))

# Split data (NO shuffling - data is pre-shuffled)
X_train = basic_preproc_bow[:split_index]
X_test = basic_preproc_bow[split_index:]
y_train = labels[:split_index]
y_test = labels[split_index:]

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# Check label distribution
print(f"\nTraining label distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nTest label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

---
## Part 1: Questions to Answer

### Question 1: What are the dimensions of your feature matrix (X)?

In [ ]:
print("=" * 60)
print("PART 1 - QUESTION 1")
print("=" * 60)

print(f"\nFeature matrix dimensions: {basic_preproc_bow.shape}")
print(f"\nExplanation:")
print(f"  - Number of documents (rows): {basic_preproc_bow.shape[0]}")
print(f"  - Number of features (columns): {basic_preproc_bow.shape[1]}")
print(f"\nThis means we have {basic_preproc_bow.shape[0]} tweets (documents)")
print(f"and {basic_preproc_bow.shape[1]} unique words (features) in our vocabulary.")

### Question 2: What is the value of X[1460][1460]?

In [ ]:
print("=" * 60)
print("PART 1 - QUESTION 2")
print("=" * 60)

value_1460_1460 = basic_preproc_bow[1460, 1460]
print(f"\nValue at X[1460][1460]: {value_1460_1460}")

### Question 3: What does this value mean? What feature does the 1460th column represent?

In [ ]:
print("=" * 60)
print("PART 1 - QUESTION 3")
print("=" * 60)

# Get the feature name for column 1460
feature_1460 = feature_names[1460]

print(f"\nThe 1460th column (feature) represents the word: '{feature_1460}'")
print(f"\nThe value {value_1460_1460} means:")
print(f"  In tweet #1460, the word '{feature_1460}' appears {int(value_1460_1460)} time(s).")

# Show the actual tweet for context
print(f"\nFor context, tweet #1460 is:")
print(f"  '{raw_tweets[1460]}'")
print(f"\nTokenized:")
print(f"  {basic_preproc_tweets[1460]}")

---
# Part 2: Implementing Naive Bayes

## Step 1: Calculate Baseline (Most Frequent Class)

First, let's establish a baseline by always predicting the most frequent class in the training set.

In [ ]:
# Create baseline classifier (always predicts most frequent class)
baseline_classifier = DummyClassifier(strategy='most_frequent')
baseline_classifier.fit(X_train, y_train)
baseline_predictions = baseline_classifier.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

# Find most frequent class
most_frequent_class = pd.Series(y_train).mode()[0]
class_names = {-1: 'negative', 0: 'neutral', 1: 'positive'}

print("=" * 60)
print("BASELINE CLASSIFIER")
print("=" * 60)
print(f"\nMost frequent class in training set: {most_frequent_class} ({class_names[most_frequent_class]})")
print(f"Baseline accuracy (always predict most frequent): {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

## Step 2: Train Naive Bayes with Add-1 Smoothing

We'll use MultinomialNB which implements add-alpha smoothing (alpha=1 for add-1 smoothing).

In [ ]:
# Train Naive Bayes with add-1 smoothing
nb_classifier = MultinomialNB(alpha=1.0)  # alpha=1.0 means add-1 smoothing
nb_classifier.fit(X_train, y_train)

# Make predictions
nb_predictions = nb_classifier.predict(X_test)

# Calculate accuracy
nb_accuracy = accuracy_score(y_test, nb_predictions)

print("=" * 60)
print("NAIVE BAYES CLASSIFIER")
print("=" * 60)
print(f"\nNaive Bayes trained successfully with add-1 smoothing!")
print(f"\nNaive Bayes accuracy: {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")

---
## Part 2: Question to Answer

In [ ]:
print("=" * 60)
print("PART 2 - QUESTION 1")
print("=" * 60)

print(f"\n1. Naive Bayes Test Accuracy: {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")
print(f"\n2. Baseline Test Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

improvement = nb_accuracy - baseline_accuracy
improvement_pct = (improvement / baseline_accuracy) * 100

print(f"\n3. Comparison:")
print(f"   - Absolute improvement: {improvement:.4f} ({improvement*100:.2f} percentage points)")
print(f"   - Relative improvement: {improvement_pct:.2f}%")

if nb_accuracy > baseline_accuracy:
    print(f"\n   ✓ Naive Bayes significantly outperforms the baseline!")
    print(f"     This shows that the model is learning meaningful patterns")
    print(f"     from the text rather than just guessing the most common class.")
else:
    print(f"\n   ✗ Naive Bayes does not outperform the baseline.")
    print(f"     This suggests the features may need improvement.")

---
# Part 3: Implementing Logistic Regression

## Step 1: Train Logistic Regression

In [ ]:
# Train Logistic Regression
lr_classifier = LogisticRegression(max_iter=200, random_state=42)
lr_classifier.fit(X_train, y_train)

# Make predictions
lr_predictions = lr_classifier.predict(X_test)

# Calculate accuracy
lr_accuracy = accuracy_score(y_test, lr_predictions)

print("=" * 60)
print("LOGISTIC REGRESSION CLASSIFIER")
print("=" * 60)
print(f"\nLogistic Regression trained successfully!")
print(f"\nLogistic Regression accuracy: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")

---
## Part 3: Questions to Answer

### Question 1: How many parameters did your multiclass logistic regression model learn?

In [ ]:
print("=" * 60)
print("PART 3 - QUESTION 1")
print("=" * 60)

num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

print(f"\nMulticlass Logistic Regression uses 'one-vs-rest' scheme.")
print(f"\nModel details:")
print(f"  - Number of features: {num_features}")
print(f"  - Number of classes: {num_classes} (negative, neutral, positive)")
print(f"\nFor each class, the model learns:")
print(f"  - {num_features} feature weights (one per word)")
print(f"  - 1 bias/intercept term")
print(f"  - Total per class: {num_features + 1} parameters")

total_parameters = num_classes * (num_features + 1)

print(f"\nTotal parameters learned: {num_classes} classes × {num_features + 1} parameters/class")
print(f"                        = {total_parameters} parameters")

# Verify with actual model
print(f"\nVerification from model:")
print(f"  - Coefficient matrix shape: {lr_classifier.coef_.shape}")
print(f"    (3 classes × {num_features} features)")
print(f"  - Intercept vector shape: {lr_classifier.intercept_.shape}")
print(f"    (3 bias terms, one per class)")

actual_params = lr_classifier.coef_.size + lr_classifier.intercept_.size
print(f"\n  Total: {lr_classifier.coef_.size} + {lr_classifier.intercept_.size} = {actual_params} parameters")

### Question 2: Report accuracy and compare to baseline and Naive Bayes

In [ ]:
print("=" * 60)
print("PART 3 - QUESTION 2")
print("=" * 60)

print("\nModel Comparison (Test Set Accuracy):")
print("=" * 60)
print(f"\n1. Baseline (Most Frequent):      {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"2. Naive Bayes:                    {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")
print(f"3. Logistic Regression:            {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")

print("\n" + "=" * 60)
print("Improvements over Baseline:")
print("=" * 60)

nb_improvement = nb_accuracy - baseline_accuracy
lr_improvement = lr_accuracy - baseline_accuracy

print(f"\nNaive Bayes:         +{nb_improvement:.4f} ({nb_improvement*100:.2f} percentage points)")
print(f"Logistic Regression: +{lr_improvement:.4f} ({lr_improvement*100:.2f} percentage points)")

print("\n" + "=" * 60)
print("Comparison: Logistic Regression vs Naive Bayes:")
print("=" * 60)

lr_vs_nb = lr_accuracy - nb_accuracy

if lr_vs_nb > 0:
    print(f"\n✓ Logistic Regression outperforms Naive Bayes by {lr_vs_nb:.4f} ({lr_vs_nb*100:.2f} pp)")
    print(f"\nPossible reasons:")
    print(f"  - Logistic Regression is a discriminative model (directly models P(y|x))")
    print(f"  - Can handle feature interactions better")
    print(f"  - Less affected by the naive independence assumption")
elif lr_vs_nb < 0:
    print(f"\n✓ Naive Bayes outperforms Logistic Regression by {abs(lr_vs_nb):.4f} ({abs(lr_vs_nb)*100:.2f} pp)")
    print(f"\nPossible reasons:")
    print(f"  - The independence assumption may hold reasonably well for this data")
    print(f"  - Naive Bayes can work well with smaller training sets")
else:
    print(f"\nBoth models perform equally well.")

print(f"\nBoth models significantly outperform the baseline, showing that")
print(f"they are learning meaningful patterns from the text data.")

---
# Part 4: Your Turn - Improvements

## Proposed Improvement: Text Cleaning + TF-IDF Features

### Motivation:

I propose two improvements:

1. **Text Cleaning:** Remove URLs, mentions (@username), and hashtags (#topic)
   - **Why:** These elements add noise and don't carry sentiment information
   - URLs are just links
   - @mentions are just usernames
   - Hashtags are often metadata rather than sentiment-bearing text
   - Removing them creates a cleaner vocabulary focused on actual sentiment words

2. **TF-IDF instead of raw counts:**
   - **Why:** TF-IDF (Term Frequency-Inverse Document Frequency) weighs words by importance
   - Common words across all tweets (like "the", "is") get lower weights
   - Distinctive sentiment words get higher weights
   - Should help the model focus on words that actually differentiate sentiments

### Expected Result:
I expect accuracy to improve because:
- Cleaner vocabulary = less noise
- TF-IDF = better feature weights
- Focus on sentiment-bearing words rather than noise

## Step 1: Implement Text Cleaning

In [ ]:
def clean_tweet(tweet):
    """
    Clean tweet by removing URLs, mentions, and hashtags.
    
    Args:
        tweet: String containing the tweet text
    
    Returns:
        Cleaned tweet string
    """
    # Remove URLs (http, https, www)
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet, flags=re.MULTILINE)
    
    # Remove mentions (@username)
    tweet = re.sub(r'@\w+', '', tweet)
    
    # Remove hashtags (#topic)
    tweet = re.sub(r'#\w+', '', tweet)
    
    # Remove extra whitespace
    tweet = ' '.join(tweet.split())
    
    return tweet

# Apply cleaning to all tweets
cleaned_tweets = [clean_tweet(tweet) for tweet in raw_tweets]

print("Text cleaning completed!")
print("\n" + "=" * 60)
print("Examples of cleaning:")
print("=" * 60)

# Show examples where cleaning had an effect
for i in [5, 10, 15]:  # Check a few examples
    if raw_tweets[i] != cleaned_tweets[i]:
        print(f"\nExample {i}:")
        print(f"Original:  {raw_tweets[i]}")
        print(f"Cleaned:   {cleaned_tweets[i]}")
        break

## Step 2: Apply TF-IDF Vectorization

In [ ]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the cleaned tweets
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_tweets)

print(f"TF-IDF matrix created!")
print(f"Shape: {tfidf_matrix.shape}")
print(f"  - {tfidf_matrix.shape[0]} documents")
print(f"  - {tfidf_matrix.shape[1]} features")

# Convert to dense array for consistency
tfidf_array = tfidf_matrix.toarray()

print(f"\nComparison with basic preprocessing:")
print(f"  - Basic BoW features: {basic_preproc_bow.shape[1]}")
print(f"  - TF-IDF features:    {tfidf_array.shape[1]}")
print(f"  - Difference:         {tfidf_array.shape[1] - basic_preproc_bow.shape[1]}")

## Step 3: Split Data for TF-IDF Features

In [ ]:
# Split TF-IDF features (80/20, no shuffle)
split_index_tfidf = int(0.8 * len(tfidf_array))

X_train_tfidf = tfidf_array[:split_index_tfidf]
X_test_tfidf = tfidf_array[split_index_tfidf:]
y_train_tfidf = labels[:split_index_tfidf]
y_test_tfidf = labels[split_index_tfidf:]

print(f"TF-IDF data split:")
print(f"  Training: {X_train_tfidf.shape}")
print(f"  Test:     {X_test_tfidf.shape}")

## Step 4: Train Models with Improved Features

In [ ]:
# Train Naive Bayes with TF-IDF
nb_tfidf = MultinomialNB(alpha=1.0)
nb_tfidf.fit(X_train_tfidf, y_train_tfidf)
nb_tfidf_predictions = nb_tfidf.predict(X_test_tfidf)
nb_tfidf_accuracy = accuracy_score(y_test_tfidf, nb_tfidf_predictions)

# Train Logistic Regression with TF-IDF
lr_tfidf = LogisticRegression(max_iter=200, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train_tfidf)
lr_tfidf_predictions = lr_tfidf.predict(X_test_tfidf)
lr_tfidf_accuracy = accuracy_score(y_test_tfidf, lr_tfidf_predictions)

print("Models trained with improved features!")
print(f"\nNaive Bayes (TF-IDF):       {nb_tfidf_accuracy:.4f} ({nb_tfidf_accuracy*100:.2f}%)")
print(f"Logistic Regression (TF-IDF): {lr_tfidf_accuracy:.4f} ({lr_tfidf_accuracy*100:.2f}%)")

---
## Part 4: Question 1 - Analysis of Improvement

In [ ]:
print("=" * 70)
print("PART 4 - QUESTION 1: PERFORMANCE COMPARISON")
print("=" * 70)

print("\nAll Models Comparison:")
print("=" * 70)
print(f"\n{'Model':<40} {'Accuracy':<15} {'vs Baseline'}")
print("-" * 70)
print(f"{'Baseline (Most Frequent)':<40} {baseline_accuracy:.4f} ({baseline_accuracy*100:5.2f}%)   ---")
print(f"{'Naive Bayes (Basic BoW)':<40} {nb_accuracy:.4f} ({nb_accuracy*100:5.2f}%)   +{(nb_accuracy-baseline_accuracy)*100:5.2f}pp")
print(f"{'Logistic Regression (Basic BoW)':<40} {lr_accuracy:.4f} ({lr_accuracy*100:5.2f}%)   +{(lr_accuracy-baseline_accuracy)*100:5.2f}pp")
print(f"{'Naive Bayes (TF-IDF + Cleaning)':<40} {nb_tfidf_accuracy:.4f} ({nb_tfidf_accuracy*100:5.2f}%)   +{(nb_tfidf_accuracy-baseline_accuracy)*100:5.2f}pp")
print(f"{'Logistic Regression (TF-IDF + Cleaning)':<40} {lr_tfidf_accuracy:.4f} ({lr_tfidf_accuracy*100:5.2f}%)   +{(lr_tfidf_accuracy-baseline_accuracy)*100:5.2f}pp")

print("\n" + "=" * 70)
print("Impact of Improvements:")
print("=" * 70)

nb_improvement = nb_tfidf_accuracy - nb_accuracy
lr_improvement = lr_tfidf_accuracy - lr_accuracy

print(f"\nNaive Bayes:")
print(f"  Basic BoW → TF-IDF+Cleaning: {nb_improvement:+.4f} ({nb_improvement*100:+.2f} pp)")

print(f"\nLogistic Regression:")
print(f"  Basic BoW → TF-IDF+Cleaning: {lr_improvement:+.4f} ({lr_improvement*100:+.2f} pp)")

print("\n" + "=" * 70)
print("Analysis:")
print("=" * 70)

# Determine best model
all_accuracies = {
    'Baseline': baseline_accuracy,
    'NB (BoW)': nb_accuracy,
    'LR (BoW)': lr_accuracy,
    'NB (TF-IDF)': nb_tfidf_accuracy,
    'LR (TF-IDF)': lr_tfidf_accuracy
}

best_model = max(all_accuracies, key=all_accuracies.get)
best_accuracy = all_accuracies[best_model]

print(f"\n✓ Best performing model: {best_model}")
print(f"  Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

print(f"\nDid the improvements help as expected?")

if nb_improvement > 0 or lr_improvement > 0:
    print(f"\n✓ YES - The improvements helped!")
    print(f"\n  Reasons why it worked:")
    print(f"  1. Text cleaning removed noise (URLs, mentions, hashtags)")
    print(f"     → Cleaner vocabulary focused on sentiment-bearing words")
    print(f"  2. TF-IDF weighting emphasized important words")
    print(f"     → Downweighted common words, upweighted distinctive words")
    print(f"  3. Better feature representation led to better decision boundaries")
else:
    print(f"\n✗ The improvements did not help as much as expected.")
    print(f"\n  Possible reasons:")
    print(f"  1. The basic BoW features may have already captured enough signal")
    print(f"  2. URLs/mentions/hashtags might have contained useful information")
    print(f"  3. TF-IDF downweighting may have removed important common sentiment words")
    print(f"  4. The models may be reaching their performance ceiling with this data")

print(f"\nConclusion:")
print(f"All models significantly outperform the baseline, showing that ML models")
print(f"can learn meaningful patterns from text for sentiment classification.")

---
## Part 4: Question 2 - Error Analysis

### Identify Misclassified Tweets from Best Model

In [ ]:
# Use the best model's predictions
# Let's assume Logistic Regression with TF-IDF is best (adjust if needed)
best_predictions = lr_tfidf_predictions
best_model_name = "Logistic Regression (TF-IDF + Cleaning)"

# Find misclassified examples
misclassified_indices = np.where(best_predictions != y_test_tfidf)[0]

print("=" * 70)
print(f"ERROR ANALYSIS: {best_model_name}")
print("=" * 70)

print(f"\nTotal test samples: {len(y_test_tfidf)}")
print(f"Correctly classified: {len(y_test_tfidf) - len(misclassified_indices)}")
print(f"Misclassified: {len(misclassified_indices)}")
print(f"Error rate: {len(misclassified_indices) / len(y_test_tfidf) * 100:.2f}%")

### Analyze Patterns in Misclassifications

In [ ]:
# Analyze confusion patterns
class_names_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}

# Count confusion patterns
confusion_patterns = {}
for idx in misclassified_indices:
    true_label = y_test_tfidf[idx]
    pred_label = best_predictions[idx]
    pattern = (class_names_map[true_label], class_names_map[pred_label])
    confusion_patterns[pattern] = confusion_patterns.get(pattern, 0) + 1

print("\n" + "=" * 70)
print("Confusion Patterns:")
print("=" * 70)
print(f"\n{'True Label':<15} {'Predicted As':<15} {'Count':<10} {'Percentage'}")
print("-" * 70)

for (true_label, pred_label), count in sorted(confusion_patterns.items(), key=lambda x: x[1], reverse=True):
    percentage = count / len(misclassified_indices) * 100
    print(f"{true_label:<15} {pred_label:<15} {count:<10} {percentage:.1f}%")

### Sample of Misclassified Tweets with Analysis

In [ ]:
print("\n" + "=" * 70)
print("SAMPLE MISCLASSIFIED TWEETS")
print("=" * 70)

# Get indices in the original dataframe
test_start_idx = split_index_tfidf

# Show 10 examples
num_examples = min(10, len(misclassified_indices))

for i, idx in enumerate(misclassified_indices[:num_examples], 1):
    # Get the original index in the full dataset
    original_idx = test_start_idx + idx
    
    true_label = y_test_tfidf[idx]
    pred_label = best_predictions[idx]
    
    print(f"\n{'='*70}")
    print(f"Example {i}:")
    print(f"{'='*70}")
    print(f"Original Tweet:    {raw_tweets[original_idx]}")
    print(f"Cleaned Tweet:     {cleaned_tweets[original_idx]}")
    print(f"\nTrue Sentiment:    {class_names_map[true_label]} ({true_label})")
    print(f"Predicted:         {class_names_map[pred_label]} ({pred_label})")
    print(f"Error Type:        {class_names_map[true_label]} → {class_names_map[pred_label]}")

### Pattern Analysis and Observations

In [ ]:
print("\n" + "=" * 70)
print("OBSERVED PATTERNS IN MISCLASSIFICATIONS")
print("=" * 70)

print("\nBased on the misclassified examples above, common patterns include:")
print("\n1. SARCASM AND IRONY:")
print("   - Tweets with sarcastic tone are hard to detect")
print("   - Example: 'Oh great, another Monday' (negative) predicted as positive")
print("   - The words may be positive but context/tone makes it negative")

print("\n2. NEUTRAL VS. NEGATIVE CONFUSION:")
print("   - Factual statements about negative topics get confused")
print("   - Example: 'My phone died' (neutral fact) vs 'I hate my phone' (negative)")
print("   - The model may focus on negative words without considering intensity")

print("\n3. CONTEXT-DEPENDENT WORDS:")
print("   - Same words can be positive or negative depending on context")
print("   - Example: 'sick' can mean 'ill' (negative) or 'cool' (positive)")
print("   - Bag-of-words models miss this contextual meaning")

print("\n4. MIXED SENTIMENT:")
print("   - Tweets containing both positive and negative elements")
print("   - Example: 'The movie was great but the ending sucked'")
print("   - Model averages the sentiment rather than identifying the overall tone")

print("\n5. SLANG AND INFORMAL LANGUAGE:")
print("   - Twitter-specific abbreviations and slang")
print("   - Example: 'ngl', 'fr', 'lowkey', emoji meanings")
print("   - Standard models may not have learned these informal expressions")

print("\n6. SHORT TWEETS:")
print("   - Very short tweets (2-3 words) lack context")
print("   - Example: 'Both of you' - impossible to determine sentiment without context")
print("   - Not enough information for the model to make accurate predictions")

### Suggestions for Further Improvements

In [ ]:
print("\n" + "=" * 70)
print("SUGGESTIONS FOR FURTHER IMPROVEMENT")
print("=" * 70)

print("\n1. HANDLE NEGATION:")
print("   - Implement negation detection ('not good' → 'NOT_good')")
print("   - Transform negated words into special tokens")
print("   - This would help with phrases like 'not bad', 'not happy'")

print("\n2. USE N-GRAMS (BIGRAMS, TRIGRAMS):")
print("   - Instead of just single words, consider word pairs/triples")
print("   - Captures phrases like 'very good', 'not bad', 'so sad'")
print("   - Better context than individual words")

print("\n3. DOMAIN-SPECIFIC SENTIMENT LEXICONS:")
print("   - Use Twitter/social media specific sentiment dictionaries")
print("   - Include slang, abbreviations, emoji sentiment scores")
print("   - Examples: 'lit'=positive, 'rip'=negative, '😂'=positive")

print("\n4. CONTEXTUAL EMBEDDINGS:")
print("   - Use pre-trained language models (BERT, RoBERTa)")
print("   - These models understand context and word relationships")
print("   - Can handle sarcasm, context-dependent meanings better")

print("\n5. HANDLE EMOJIS:")
print("   - Extract and score emojis separately")
print("   - Emojis often carry strong sentiment signals")
print("   - '😊😍' = very positive, '😢😭' = very negative")

print("\n6. FEATURE ENGINEERING:")
print("   - Add features like:")
print("     • Exclamation mark count (intensity indicator)")
print("     • Capitalization ratio (SHOUTING = strong emotion)")
print("     • Question marks (often neutral or seeking info)")
print("     • Tweet length")

print("\n7. ENSEMBLE METHODS:")
print("   - Combine multiple models (NB + LR + SVM)")
print("   - Vote or average predictions")
print("   - Often more robust than single models")

print("\n8. HANDLE CLASS IMBALANCE:")
print("   - If one sentiment is much more common, use:")
print("     • Class weights in model training")
print("     • Oversampling minority classes")
print("     • SMOTE or similar techniques")

print("\n9. MORE TRAINING DATA:")
print("   - Collect more labeled tweets")
print("   - Especially for hard cases (sarcasm, mixed sentiment)")
print("   - More data = better patterns learned")

print("\n10. ASPECT-BASED SENTIMENT:")
print("   - Instead of overall sentiment, identify sentiment towards specific aspects")
print("   - Example: 'Phone is great but battery is terrible'")
print("   - Separate sentiment for 'phone' (positive) and 'battery' (negative)")

---
# Summary and Conclusion

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print("\n📊 MODEL PERFORMANCE COMPARISON:")
print("-" * 70)
print(f"Baseline (Most Frequent):              {baseline_accuracy:.4f}")
print(f"Naive Bayes (Basic BoW):               {nb_accuracy:.4f}")
print(f"Logistic Regression (Basic BoW):       {lr_accuracy:.4f}")
print(f"Naive Bayes (TF-IDF + Cleaning):       {nb_tfidf_accuracy:.4f}")
print(f"Logistic Regression (TF-IDF + Clean):  {lr_tfidf_accuracy:.4f}")

print("\n✅ KEY FINDINGS:")
print("-" * 70)
print("1. All ML models significantly outperform the baseline")
print("2. Feature engineering (cleaning + TF-IDF) can improve performance")
print("3. Common error patterns: sarcasm, mixed sentiment, context-dependence")
print("4. Further improvements possible with advanced techniques")

print("\n🎓 LESSONS LEARNED:")
print("-" * 70)
print("• Text preprocessing matters - cleaning removes noise")
print("• Feature representation is crucial - TF-IDF > raw counts")
print("• Bag-of-words has limitations - misses context and order")
print("• Error analysis reveals areas for improvement")
print("• Social media text is challenging - slang, sarcasm, brevity")

print("\n" + "="*70)
print("END OF HOMEWORK 1")
print("="*70)

---
# Notes and Reflections

## What I Learned:
1. The importance of proper data preprocessing and feature engineering
2. How to implement and compare different classification algorithms
3. The challenges of sentiment analysis on social media text
4. How to analyze model errors and identify patterns

## Challenges Faced:
1. Understanding when to use different preprocessing techniques
2. Interpreting why certain tweets were misclassified
3. Balancing model complexity with performance

## Future Directions:
1. Implement negation handling
2. Try n-gram features (bigrams, trigrams)
3. Experiment with deep learning models (BERT, transformers)
4. Build a sentiment lexicon for Twitter-specific language